In [8]:
# Fiks path for imports
import sys
from pathlib import Path
repo_root = Path().resolve().parents[1]
sys.path.append(str(repo_root))

import torch

import matplotlib.pyplot as plt


from hesel_scraper.bout_dump import BOUTHESELInfo
from hesel_scraper.bout_phys import BOUTHESELPhys


root = repo_root / r"sim_data/data_25_512_Alexander"
info = BOUTHESELInfo(root)
phys = BOUTHESELPhys(info)

# Define the spatial grid
num_x = info.parameters.num_x
num_t = info.parameters.num_t
alpha = 0.0
beta = info.parameters.Lx
x = torch.linspace(alpha, beta, num_x, device=info.device, dtype=info.dtype)
t = torch.arange(0, num_t, device=info.device, dtype=info.dtype)*info.parameters.dt

# force profiles:
L_alpha = L_beta = 0.5
L_t = 50

sigma_alpha = 1-torch.exp(-(x - alpha) / L_alpha)
sigma_beta = 1-torch.exp(-(beta - x) / L_beta)
sigma_t = 1-torch.exp(-t / L_t)
D_x = sigma_alpha**2 * sigma_beta**2

In [9]:
# Plot forcing profiles in two subplots
t_idxses = 50
x_plot = x.detach().cpu()

sigma_alpha_plot = (sigma_alpha.detach().cpu())
sigma_beta_plot = (sigma_beta.detach().cpu())
D_x = sigma_alpha_plot**2 * sigma_beta_plot**2

# Use index on x-axis for temporal plot: 0..t_idxses (inclusive)
t_plot = torch.arange(t_idxses + 1)
sigma_t_plot = (sigma_t.detach().cpu()[: t_idxses + 1])

fig, axes = plt.subplots(2, 1, figsize=(10, 8), dpi=2000, sharex=False)

# Top subplot: sigma_alpha(x) and sigma_beta(x)
axes[0].plot(x_plot, sigma_alpha_plot, label=r"$\sigma_\alpha(x)$", linewidth=2)
axes[0].plot(x_plot, sigma_beta_plot, label=r"$\sigma_\beta(x)$", linewidth=2)
axes[0].plot(x_plot, D_x, label=r"$D_x = \sigma_\alpha(x)^2 \sigma_\beta(x)^2$", linewidth=2, linestyle="--", color="tab:purple")
axes[0].set_xlabel("x")
axes[0].set_ylabel("Forcing")
axes[0].set_title(r"Spatial forcing profiles: $\sigma_\alpha(x)$ and $\sigma_\beta(x)$")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Bottom subplot: first points of sigma_t(t) with index axis
axes[1].plot(t_plot, sigma_t_plot, color="tab:green", label=rf"$\sigma_t(t)^2$ (0..{t_idxses})", linewidth=2)
axes[1].set_xlim(0, t_idxses)
axes[1].set_xlabel("time index")
axes[1].set_ylabel("Forcing")
axes[1].set_title(rf"Temporal forcing profile: $\sigma_t(t)$ for indices 0..{t_idxses}")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()